CMMD

In [ ]:
import os
import numpy as np
import pandas as pd
import pydicom as pdcm
import cv2

def np_CountUpContinuingOnes(b_arr):
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)
    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]
    return right - left - 1

def ExtractBreast(img):
    img_copy = img.copy()
    img = np.where(img <= 20, 0, img)
    height, _ = img.shape
    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    img = img[:, col_ind]
    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]
    return img_copy[row_ind][:, col_ind]

# 读取数据划分CSV文件
split_csv_path = "../classification_data/classification_split.csv"
split_df = pd.read_csv(split_csv_path)
# 只保留CMMD数据集
split_df = split_df[split_df['dataset'] == 'CMMD']

# 读取XLSX文件
xlsx_path = '/Volumes/CMMD/CMMD_clinicaldata_revision.xlsx'
df = pd.read_excel(xlsx_path)
df['subtype'] = df['subtype'].fillna('-')

# 定义输入和输出路径
DATA_PATH = "/Volumes/CMMD/manifest-1616439774456/CMMD"
OUTPUT_BASE_PATH = "../classification_data/CMMD"

def process_and_save():
    for index, row in df.iterrows():
        folder_name = row['ID1']
        left_right = str(row['LeftRight']).replace(' ', '')
        abnormality = str(row['abnormality']).replace(' ','')
        if abnormality=='mass':
            abnormality=['Mass']
        elif abnormality=='calcification':
            abnormality=['Calcification']
        elif abnormality=='both':
            abnormality=['Calcification','Mass']
        classification = str(row['classification']).replace(' ', '')
        subtype = str(row['subtype']).replace(' ', '')
        meta_data = {
            'Finding': abnormality,
            'Pathology': classification,
            'Subtype': subtype
        }
        if meta_data['Subtype'] == '-':
            del meta_data['Subtype']

        folder_path = os.path.join(DATA_PATH, folder_name)
        if os.path.exists(folder_path):
            dcm_files = []
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    if file.endswith('.dcm') and not file.startswith('._'):
                        dcm_files.append(os.path.join(root, file))

            if len(dcm_files) not in [2, 4]:
                print(f"Expected 2 or 4 DICOM files in {folder_path}, but found {len(dcm_files)}")
                continue

            if len(dcm_files) == 2:
                for i, dcm_file in enumerate(dcm_files):
                    dcm = pdcm.dcmread(dcm_file)
                    img = dcm.pixel_array
                    img = ExtractBreast(img)
                    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
                    output_subdir = f"{folder_name}-{left_right}-{i + 1}"
                    
                    # 查找对应的data_split
                    split_info = split_df[split_df['data_name'] == output_subdir]
                    if split_info.empty:
                        print(f"No split info found for {output_subdir}")
                        continue
                    
                    data_split = split_info['data_split'].values[0]
                    output_dir = os.path.join(OUTPUT_BASE_PATH, data_split, output_subdir)
                    os.makedirs(output_dir, exist_ok=True)
                    
                    jpg_path = os.path.join(output_dir, 'img.jpg')
                    cv2.imwrite(jpg_path, img)
                    npy_path = os.path.join(output_dir, 'info_dict.npy')
                    np.save(npy_path, meta_data)
                    print(f"Processed {output_subdir} for {data_split} set")

            elif len(dcm_files) == 4:
                relevant_files = ['1-1.dcm', '1-2.dcm'] if left_right == 'L' else ['1-3.dcm', '1-4.dcm']
                for dcm_file in dcm_files:
                    if os.path.basename(dcm_file) in relevant_files:
                        dcm = pdcm.dcmread(dcm_file)
                        img = dcm.pixel_array
                        img = ExtractBreast(img)
                        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
                        file_index = relevant_files.index(os.path.basename(dcm_file)) + 1
                        output_subdir = f"{folder_name}-{left_right}-{file_index}"
                        
                        # 查找对应的data_split
                        split_info = split_df[split_df['data_name'] == output_subdir]
                        if split_info.empty:
                            print(f"No split info found for {output_subdir}")
                            continue
                        
                        data_split = split_info['data_split'].values[0]
                        output_dir = os.path.join(OUTPUT_BASE_PATH, data_split, output_subdir)
                        os.makedirs(output_dir, exist_ok=True)
                        
                        jpg_path = os.path.join(output_dir, 'img.jpg')
                        cv2.imwrite(jpg_path, img)
                        npy_path = os.path.join(output_dir, 'info_dict.npy')
                        np.save(npy_path, meta_data)
                        print(f"Processed {output_subdir} for {data_split} set")
        else:
            print(f"Folder {folder_name} not found.")

# 处理并保存数据
process_and_save()
print("Processing complete.")